water diffusion coefficient was calculated using matplotlib, the first 10 ns of each temperature step was excluded to allow for equilibration. The diffusion coefficient was calculated using the Einstein relation, which relates the mean squared displacement (MSD) of particles to time. 

In [ ]:
import os
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.ticker import AutoMinorLocator


# =============================================================================
# INPUT / OUTPUT PATHS
# =============================================================================

INPUT_ROOT = Path("./input").resolve()
OUTPUT_ROOT = Path(
    "./output/fig-s3"
).resolve()

DTRAJ_DIR = (
    INPUT_ROOT
    / "cooling"
    / "diagnostics"
    / "dtrajs"
)

DIFFUSION_CSV = (
    INPUT_ROOT
    / "cooling"
    / "diagnostics"
    / "diffusion"
    / "diffusion-coefficients.csv"
)

OUTPUT_PDF = (
    OUTPUT_ROOT
    / "self_diffusion_coefficients_kl_changes_during_cooling.pdf"
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

DTRAJ_FILES = [
    DTRAJ_DIR
    / f"split{i}_final_npy_t_250_k_50.h5"
    for i in range(15)
]


# =============================================================================
# VALIDATE INPUT FILES
# =============================================================================

missing_files = [
    path
    for path in DTRAJ_FILES
    if not path.is_file()
]

if not DIFFUSION_CSV.is_file():
    missing_files.append(
        DIFFUSION_CSV
    )

if missing_files:
    missing_text = "\n".join(
        f"  - {path}"
        for path in missing_files
    )

    raise FileNotFoundError(
        "Required input files are missing:\n"
        f"{missing_text}"
    )

print("Input directory:")
print(f"  {INPUT_ROOT}")

print("\nDiscrete trajectories:")
print(f"  {DTRAJ_DIR}")

print("\nDiffusion coefficients:")
print(f"  {DIFFUSION_CSV}")


# =============================================================================
# ANNEALING SCHEDULE
# =============================================================================

annealing_time_ps = np.array(
    [
        0, 1,
        15625, 15626,
        31250, 31251,
        46875, 46876,
        62500, 62501,
        78125, 78126,
        93750, 93751,
        109375, 109376,
        125000, 125001,
        140625, 140626,
        156250, 156251,
        171875, 171876,
        187500, 187501,
        203125, 203126,
        218750, 218751,
        234375, 234376,
        250000, 250001,
        265625, 265626,
        281250, 281251,
        296875, 296876,
        312500, 312501,
        328125, 328126,
        343750, 343751,
        359375, 359376,
        375000, 375001,
        390625, 390626,
        406250, 406251,
        421875, 421876,
        437500, 437501,
        453125, 453126,
        468750, 468751,
        484375, 484376,
        500000, 500001,
        515625, 515626,
        531250, 531251,
        546875, 546876,
        562500, 562501,
        578125, 578126,
        593750, 593751,
        609375, 609376,
        625000, 625001,
        640625, 640626,
        656250, 656251,
        671875, 671876,
        687500, 687501,
        703125, 703126,
        718750, 718751,
        734375, 734376,
        750000, 750001,
        765625, 765626,
        781250, 781251,
        796875, 796876,
        812500, 812501,
        828125, 828126,
        843750, 843751,
        859375, 859376,
        875000, 875001,
        890625, 890626,
        906250, 906251,
        921875, 921876,
        937500, 937501,
        953125, 953126,
        968750, 968751,
        984375, 984376,
        1000000, 1000001,
        1015625, 1015626,
        1031250, 1031251,
        1046875, 1046876,
        1062500, 1062501,
        1078125, 1078126,
        1093750, 1093751,
        1109375, 1109376,
        1125000, 1125001,
        1140625, 1140626,
        1156250, 1156251,
        1171875, 1171876,
        1187500, 1187501,
        1203125, 1203126,
        1218750, 1218751,
        1234375, 1234376,
        1250000, 1250001,
    ],
    dtype=float,
)

annealing_temp_K = np.array(
    [
        277, 276, 276, 275, 275, 274, 274, 273, 273, 272, 272, 271, 271,
        270, 270, 269, 269, 268, 268, 267, 267, 266, 266, 265, 265, 264,
        264, 263, 263, 262, 262, 261, 261, 260, 260, 259, 259, 258, 258,
        257, 257, 256, 256, 255, 255, 254, 254, 253, 253, 252, 252, 251,
        251, 250, 250, 249, 249, 248, 248, 247, 247, 246, 246, 245, 245,
        244, 244, 243, 243, 242, 242, 241, 241, 240, 240, 239, 239, 238,
        238, 237, 237, 236, 236, 235, 235, 234, 234, 233, 233, 232, 232,
        231, 231, 230, 230, 229, 229, 228, 228, 227, 227, 226, 226, 225,
        225, 224, 224, 223, 223, 222, 222, 221, 221, 220, 220, 219, 219,
        218, 218, 217, 217, 216, 216, 215, 215, 214, 214, 213, 213, 212,
        212, 211, 211, 210, 210, 209, 209, 208, 208, 207, 207, 206, 206,
        205, 205, 204, 204, 203, 203, 202, 202, 201, 201, 200, 200, 199,
        199, 198, 198, 197, 197, 196,
    ],
    dtype=float,
)

if len(annealing_time_ps) != len(annealing_temp_K):
    raise RuntimeError(
        "Annealing time and temperature arrays have different lengths: "
        f"{len(annealing_time_ps)} vs {len(annealing_temp_K)}."
    )


# =============================================================================
# LOAD DISCRETE TRAJECTORIES
# =============================================================================

def load_all_arrays_multi(filenames):
    arrays = {}

    for split_index, filename in enumerate(filenames):
        split_label = f"split{split_index}"

        print(f"Loading {filename.name}")

        with h5py.File(filename, "r") as handle:
            n_datasets = 0

            for name, obj in handle.items():
                if isinstance(obj, h5py.Dataset):
                    data = obj[...]

                    key = f"{split_label}/{name}"
                    arrays[key] = data
                    n_datasets += 1

            print(f"  {n_datasets} datasets")

    if not arrays:
        raise RuntimeError(
            "No HDF5 datasets were loaded."
        )

    return arrays


# =============================================================================
# KL-DIVERGENCE CALCULATION
# =============================================================================

def make_time_axis(
    indices,
    dt=1.0,
    t0=0.0,
):
    indices = np.asarray(
        indices,
        dtype=float,
    )

    return t0 + dt * indices


def state_population_distribution_window(
    arrays_dict,
    start_frame,
    window_size,
    require_full_window=True,
):
    counts = {}
    total = 0

    for arr in arrays_dict.values():
        arr = np.asarray(arr)

        n_frames = arr.shape[0]

        if (
            require_full_window
            and n_frames < start_frame + window_size
        ):
            continue

        if n_frames <= start_frame:
            continue

        end_frame = min(
            n_frames,
            start_frame + window_size,
        )

        window = arr[
            start_frame:end_frame
        ]

        values, counts_local = np.unique(
            window,
            return_counts=True,
        )

        for value, count in zip(
            values,
            counts_local,
        ):
            state_value = int(value)

            counts[state_value] = (
                counts.get(
                    state_value,
                    0,
                )
                + int(count)
            )

        total += window.size

    if total == 0:
        return {}, 0

    population = {
        state_value: count / total
        for state_value, count
        in counts.items()
    }

    return population, total


def kl_divergence_distributions(
    p,
    q,
    eps=1e-12,
):
    all_states = sorted(
        set(p.keys())
        | set(q.keys())
    )

    if not all_states:
        return np.nan

    p_vec = np.array(
        [
            p.get(
                state_value,
                0.0,
            )
            for state_value
            in all_states
        ],
        dtype=float,
    )

    q_vec = np.array(
        [
            q.get(
                state_value,
                0.0,
            )
            for state_value
            in all_states
        ],
        dtype=float,
    )

    p_vec += eps
    q_vec += eps

    p_vec /= p_vec.sum()
    q_vec /= q_vec.sum()

    kl = np.sum(
        p_vec
        * np.log(
            p_vec / q_vec
        )
    )

    return float(kl)


def calculate_kl_over_windows_against_reference(
    arrays_dict,
    ref_frame_index,
    window_size=10,
    stride=None,
    start_frame=0,
    end_frame=None,
    dt=1.0,
    t0=0.0,
):
    if stride is None:
        stride = window_size

    ref_start = max(
        0,
        ref_frame_index
        - window_size
        + 1,
    )

    ref_end = (
        ref_start
        + window_size
    )

    filtered = {
        key: value
        for key, value
        in arrays_dict.items()
        if np.asarray(
            value
        ).shape[0] >= ref_end
    }

    if not filtered:
        raise RuntimeError(
            "No trajectories are long enough to cover "
            f"the reference window [{ref_start}, {ref_end})."
        )

    lengths = [
        np.asarray(
            value
        ).shape[0]
        for value
        in filtered.values()
    ]

    min_len = min(lengths)

    max_valid_start = (
        min_len
        - window_size
    )

    if end_frame is None:
        end_frame = ref_frame_index

    end_frame = min(
        end_frame,
        max_valid_start,
    )

    start_frame = max(
        0,
        start_frame,
    )

    if start_frame > end_frame:
        raise ValueError(
            "start_frame must be <= end_frame."
        )

    ref_pop, ref_samples = (
        state_population_distribution_window(
            filtered,
            ref_start,
            window_size,
            require_full_window=True,
        )
    )

    print(
        "\nReference population window:"
    )

    print(
        f"  frames: "
        f"[{ref_start}, {ref_start + window_size})"
    )

    print(
        f"  samples: {ref_samples}"
    )

    for state_value in sorted(ref_pop):
        print(
            f"  state {state_value}: "
            f"{ref_pop[state_value]:.6f}"
        )

    window_starts = np.arange(
        start_frame,
        end_frame + 1,
        stride,
        dtype=int,
    )

    kl_values = np.empty(
        len(window_starts),
        dtype=float,
    )

    for index, start_value in enumerate(
        window_starts
    ):
        pop_s, n_samples_s = (
            state_population_distribution_window(
                filtered,
                start_value,
                window_size,
                require_full_window=True,
            )
        )

        if n_samples_s == 0:
            kl_values[index] = np.nan
        else:
            kl_values[index] = (
                kl_divergence_distributions(
                    pop_s,
                    ref_pop,
                )
            )

    window_centers = (
        window_starts
        + window_size / 2.0
    )

    x_values = make_time_axis(
        window_centers,
        dt=dt,
        t0=t0,
    )

    return {
        "window_starts": window_starts,
        "window_centers": window_centers,
        "x_values": x_values,
        "kl_values": kl_values,
        "ref_pop": ref_pop,
        "ref_samples": ref_samples,
        "ref_start": ref_start,
        "ref_end": ref_end,
        "window_size": window_size,
        "stride": stride,
        "dt": dt,
        "t0": t0,
    }


# =============================================================================
# CALCULATE KL DIVERGENCE
# =============================================================================

arrays_raw = load_all_arrays_multi(
    DTRAJ_FILES
)

lengths_all = [
    np.asarray(
        value
    ).shape[0]
    for value
    in arrays_raw.values()
]

print(
    "\nTrajectory lengths:"
)

print(
    f"  minimum: {min(lengths_all)}"
)

print(
    f"  maximum: {max(lengths_all)}"
)

ref_frame_index = 12500
window_size = 10
kl_dt = 1.0

kl_results = (
    calculate_kl_over_windows_against_reference(
        arrays_raw,
        ref_frame_index=ref_frame_index,
        window_size=window_size,
        stride=window_size,
        start_frame=0,
        end_frame=ref_frame_index,
        dt=kl_dt,
        t0=0.0,
    )
)


# =============================================================================
# LOAD DIFFUSION COEFFICIENT DATA
# =============================================================================

df = pd.read_csv(
    DIFFUSION_CSV,
    index_col=0,
)

if "Average" not in df.columns:
    raise ValueError(
        f"{DIFFUSION_CSV} does not contain "
        "the required 'Average' column."
    )

df.index = pd.to_numeric(
    df.index,
    errors="raise",
)

universe_cols = [
    column
    for column
    in df.columns
    if column != "Average"
]

if not universe_cols:
    raise ValueError(
        "The diffusion CSV contains no individual "
        "replicate/universe columns."
    )

for column in df.columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="raise",
    )

std_dev = df[
    universe_cols
].std(
    axis=1
)


# =============================================================================
# PLOTTING HELPERS
# =============================================================================

def format_axes(
    ax,
    minor_ticks_x=2,
    minor_ticks_y=2,
    label_fontsize=10,
    tick_labelsize=10,
    title_fontsize=10,
    stroke_in=0.01,
):
    line_width_pts = (
        stroke_in * 72
    )

    ax.minorticks_on()

    if ax.get_xscale() == "log":
        ax.xaxis.set_minor_locator(
            ticker.LogLocator(
                subs="auto"
            )
        )
    else:
        ax.xaxis.set_minor_locator(
            AutoMinorLocator(
                minor_ticks_x
            )
        )

    if ax.get_yscale() == "log":
        ax.yaxis.set_minor_locator(
            ticker.LogLocator(
                subs="auto"
            )
        )
    else:
        ax.yaxis.set_minor_locator(
            AutoMinorLocator(
                minor_ticks_y
            )
        )

    for spine in ax.spines.values():
        spine.set_linewidth(
            1.5
        )

        spine.set_color(
            "black"
        )

    ax.tick_params(
        which="major",
        width=line_width_pts,
        length=4.5,
        direction="in",
        top=True,
        bottom=True,
        left=True,
        right=True,
    )

    ax.tick_params(
        which="minor",
        width=line_width_pts * 0.75,
        length=2.25,
        direction="in",
        top=True,
        bottom=True,
        left=True,
        right=True,
    )

    ax.xaxis.label.set_size(
        label_fontsize
    )

    ax.yaxis.label.set_size(
        label_fontsize
    )

    for tick_label in (
        ax.get_xticklabels()
        + ax.get_yticklabels()
    ):
        tick_label.set_fontsize(
            tick_labelsize
        )

    ax.title.set_fontsize(
        title_fontsize
    )


def set_temperature_ticks_from_schedule(
    ax,
    temperatures_to_show,
    annealing_temp_K,
    annealing_time_ps,
    frame_dt_ps=100.0,
    frame_t0_ps=0.0,
):
    x_min, x_max = (
        ax.get_xlim()
    )

    tick_positions = []
    tick_labels = []

    unique_schedule_temps = (
        np.unique(
            annealing_temp_K
        )
    )

    for target_temp in (
        temperatures_to_show
    ):
        if (
            target_temp
            not in unique_schedule_temps
        ):
            continue

        matching = np.where(
            annealing_temp_K
            == target_temp
        )[0]

        first_match = (
            matching[0]
        )

        time_ps = (
            annealing_time_ps[
                first_match
            ]
        )

        x_position = (
            (
                time_ps
                - frame_t0_ps
            )
            / frame_dt_ps
        )

        if (
            x_min
            <= x_position
            <= x_max
            or
            x_max
            <= x_position
            <= x_min
        ):
            tick_positions.append(
                x_position
            )

            tick_labels.append(
                f"{int(target_temp)}"
            )

    ax.set_xticks(
        tick_positions
    )

    ax.set_xticklabels(
        tick_labels
    )


# =============================================================================
# FIXED FIGURE LAYOUT
# =============================================================================

DATA_MARGIN_LEFT_IN = 1.25
DATA_MARGIN_RIGHT_IN = 0.25
DATA_MARGIN_BOTTOM_IN = 0.55
DATA_MARGIN_TOP_IN = 0.35
INTER_AXES_GAP_IN = 0.75


def make_fixed_stacked_axes_figure(
    data_width,
    top_data_height,
    bottom_data_height,
    gap_height=INTER_AXES_GAP_IN,
):
    fig_width = (
        DATA_MARGIN_LEFT_IN
        + data_width
        + DATA_MARGIN_RIGHT_IN
    )

    fig_height = (
        DATA_MARGIN_BOTTOM_IN
        + bottom_data_height
        + gap_height
        + top_data_height
        + DATA_MARGIN_TOP_IN
    )

    fig = plt.figure(
        figsize=(
            fig_width,
            fig_height,
        )
    )

    ax_left = (
        DATA_MARGIN_LEFT_IN
        / fig_width
    )

    ax_width = (
        data_width
        / fig_width
    )

    bottom_ax_bottom = (
        DATA_MARGIN_BOTTOM_IN
        / fig_height
    )

    bottom_ax_height = (
        bottom_data_height
        / fig_height
    )

    top_ax_bottom = (
        DATA_MARGIN_BOTTOM_IN
        + bottom_data_height
        + gap_height
    ) / fig_height

    top_ax_height = (
        top_data_height
        / fig_height
    )

    ax_top = fig.add_axes(
        [
            ax_left,
            top_ax_bottom,
            ax_width,
            top_ax_height,
        ]
    )

    ax_bottom = fig.add_axes(
        [
            ax_left,
            bottom_ax_bottom,
            ax_width,
            bottom_ax_height,
        ]
    )

    return (
        fig,
        ax_top,
        ax_bottom,
    )


# =============================================================================
# CREATE FIGURE
# =============================================================================

fig, ax_diff, ax_kl = (
    make_fixed_stacked_axes_figure(
        data_width=8.0,
        top_data_height=3.2,
        bottom_data_height=3.2,
    )
)


# =============================================================================
# PANEL A: WATER SELF-DIFFUSION
# =============================================================================

ax_diff.errorbar(
    df.index,
    df["Average"],
    yerr=std_dev,
    fmt="o-",
    color="black",
    linewidth=1,
    capsize=3,
    markersize=3,
)

ax_diff.set_xlabel(
    "Temperature (K)"
)

ax_diff.set_ylabel(
    "Self Diffusion Coefficient"
)

ax_diff.grid(
    True,
    which="both",
    alpha=0.2,
)

format_axes(
    ax_diff,
    minor_ticks_x=2,
    minor_ticks_y=2,
    label_fontsize=10,
    tick_labelsize=10,
    title_fontsize=10,
    stroke_in=0.01,
)


# =============================================================================
# PANEL A INSET: LOG-SCALE WATER SELF-DIFFUSION
# =============================================================================

ax_diff_inset = ax_diff.inset_axes(
    [0.07, 0.29, 0.46, 0.64]
)

ax_diff_inset.errorbar(
    df.index,
    df["Average"],
    yerr=std_dev,
    fmt="o-",
    color="black",
    linewidth=1,
    capsize=3,
    markersize=3,
)

ax_diff_inset.set_yscale(
    "log"
)

ax_diff_inset.grid(
    True,
    which="both",
    alpha=0.2,
)

format_axes(
    ax_diff_inset,
    minor_ticks_x=2,
    minor_ticks_y=2,
    label_fontsize=10,
    tick_labelsize=10,
    title_fontsize=10,
    stroke_in=0.01,
)

# No axis titles in the inset.
ax_diff_inset.set_xlabel("")
ax_diff_inset.set_ylabel("")

# Match the main panel's temperature range exactly.
ax_diff_inset.set_xlim(
    ax_diff.get_xlim()
)

# Logarithmic y-axis shown explicitly as powers of ten.
ax_diff_inset.yaxis.set_major_locator(
    ticker.LogLocator(
        base=10.0,
        numticks=6,
    )
)

ax_diff_inset.yaxis.set_major_formatter(
    ticker.LogFormatterMathtext(
        base=10.0
    )
)

ax_diff_inset.yaxis.set_minor_locator(
    ticker.LogLocator(
        base=10.0,
        subs=np.arange(2, 10) * 0.1,
        numticks=12,
    )
)

ax_diff_inset.yaxis.set_minor_formatter(
    ticker.NullFormatter()
)


# =============================================================================
# PANEL B: KL DIVERGENCE
# =============================================================================

ax_kl.plot(
    kl_results[
        "x_values"
    ],
    kl_results[
        "kl_values"
    ],
)

ax_kl.set_xlabel(
    "Temperature (K)"
)

ax_kl.set_ylabel(
    "KL Divergence to 196 K ensemble"
)

vline_frame = 7340

vline_x = (
    kl_results["t0"]
    + kl_results["dt"]
    * vline_frame
)

ax_kl.axvline(
    vline_x,
    linestyle="--",
)

format_axes(
    ax_kl,
    minor_ticks_x=2,
    minor_ticks_y=2,
    label_fontsize=10,
    tick_labelsize=10,
    title_fontsize=10,
    stroke_in=0.01,
)

set_temperature_ticks_from_schedule(
    ax_kl,
    temperatures_to_show=[
        270,
        260,
        250,
        240,
        230,
        220,
        210,
        200,
    ],
    annealing_temp_K=annealing_temp_K,
    annealing_time_ps=annealing_time_ps,
    frame_dt_ps=100.0,
    frame_t0_ps=0.0,
)

ax_kl.invert_xaxis()


# =============================================================================
# PANEL LABELS
# =============================================================================

diff_pos = (
    ax_diff.get_position()
)

kl_pos = (
    ax_kl.get_position()
)

fig.text(
    diff_pos.x0 - 0.045,
    diff_pos.y1 + 0.01,
    "A",
    fontsize=14,
    ha="left",
    va="bottom",
)

fig.text(
    kl_pos.x0 - 0.045,
    kl_pos.y1 + 0.01,
    "B",
    fontsize=14,
    ha="left",
    va="bottom",
)


# =============================================================================
# SAVE
# =============================================================================

fig.savefig(
    OUTPUT_PDF
)

print(
    f"\nSaved:\n  {OUTPUT_PDF}"
)

plt.show()